# 01 — Exploratory Data Analysis

Exploratory analysis of the 2,000 synthetic patients used for the VALVE-WATCH proof of concept.

The synthetic cohort is used to test the data pipeline and software behavior, not to establish clinical performance.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_FILE = Path('../data/processed/synthetic_2000_patients.json')

with DATA_FILE.open('r', encoding='utf-8') as f:
    patients = json.load(f)

print(f'Patients: {len(patients)}')

In [ ]:
def first_item(value):
    if isinstance(value, list):
        return value[0] if value else {}
    if isinstance(value, dict):
        return value
    return {}

def number(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan

rows = []

for patient in patients:
    demographics = patient.get('demographics_and_body_size', {})
    procedures = first_item(patient.get('aortic_valve_procedures', []))
    echoes = patient.get('echocardiograms', [])
    echo = first_item(echoes)

    rows.append({
        'patient_id': patient.get('patient_id'),
        'age': number(demographics.get('age')),
        'bmi': number(demographics.get('bmi')),
        'mean_gradient': number(echo.get('mean_gradient')),
        'eoa': number(echo.get('eoa')),
        'dvi': number(echo.get('dvi')),
        'lvef': number(echo.get('lvef')),
        'num_echos': len(echoes) if isinstance(echoes, list) else 0,
    })

df = pd.DataFrame(rows)
df.head()

In [ ]:
df.describe(include='all').T

In [ ]:
numeric_cols = [
    'age', 'bmi', 'mean_gradient',
    'eoa', 'dvi', 'lvef', 'num_echos'
]

df[numeric_cols].hist(figsize=(14, 10))
plt.tight_layout()
plt.show()

In [ ]:
missingness = df.isna().mean().sort_values(ascending=False)
missingness

## Interpretation

The EDA describes the synthetic cohort and identifies missingness, distributional characteristics and available echo information. Clinical conclusions are not drawn from the synthetic data.